In [1]:
HABITICA_USER_ID = "480ac1e2-5e32-475d-bb29-71062e9b96b7"
HABITICA_API_TOKEN = "6fbbadba-c756-43ce-9327-0f94942e300b"

LLM_MODEL = "deepseek-chat"
LLM_TEMPERATURE = 0.7
LLM_API_KEY = "sk-9b666d60f284488aaaa69850226bb46a"
LLM_BASE_URL = "https://api.deepseek.ai/v1"
CHECK_INTERVAL_HOURS=24 # 定时脚本运行间隔

In [2]:
from datetime import datetime
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field, model_validator

TaskType = Literal["habit", "daily", "todo", "reward"]
AttributeType = Literal["str", "int", "per", "con"]
FrequencyType = Literal["daily", "weekly", "monthly", "yearly"]

ALLOWED_PRIORITIES = (0.1, 1.0, 1.5, 2.0)


class ChecklistItem(BaseModel):
    text: str
    completed: bool = False


class ReminderItem(BaseModel):
    id: str
    startDate: str
    time: str


class TaskData(BaseModel):
    text: str
    type: TaskType
    tags: List[str] = Field(default_factory=list)
    alias: Optional[str] = None
    attribute: Optional[AttributeType] = None
    checklist: List[ChecklistItem] = Field(default_factory=list)
    collapseChecklist: bool = False
    notes: Optional[str] = None
    date: Optional[datetime] = None
    priority: float = Field(default=1.0, description="任务优先级，允许值为 0.1, 1, 1.5, 2")
    reminders: List[ReminderItem] = Field(default_factory=list)
    frequency: FrequencyType = "weekly"
    repeat: Dict[str, bool] = Field(default_factory=dict)
    everyX: int = 1
    streak: int = 0
    daysOfMonth: List[int] = Field(default_factory=list)
    weeksOfMonth: List[int] = Field(default_factory=list)
    startDate: Optional[datetime] = None
    up: bool = True
    down: bool = True
    value: float = 0

    @model_validator(mode="after")
    def validate_task_rules(self) -> "TaskData":
        if self.priority not in ALLOWED_PRIORITIES:
            raise ValueError("priority 仅允许 0.1, 1, 1.5, 2")

        if self.type != "todo" and self.date is not None:
            raise ValueError("date 仅适用于 todo 类型")

        if self.type != "daily":
            if self.streak != 0:
                raise ValueError("streak 仅适用于 daily 类型")
            if self.daysOfMonth:
                raise ValueError("daysOfMonth 仅适用于 daily 类型")
            if self.weeksOfMonth:
                raise ValueError("weeksOfMonth 仅适用于 daily 类型")
            if self.startDate is not None:
                raise ValueError("startDate 仅适用于 daily 类型")

        if self.type != "habit" and (self.up is not True or self.down is not True):
            raise ValueError("up/down 仅适用于 habit 类型")

        if self.type != "reward" and self.value != 0:
            raise ValueError("value 仅适用于 reward 类型")

        if self.value < 0:
            raise ValueError("value 不能小于 0")

        return self

    def to_dict(self) -> Dict[str, Any]:
        return self.model_dump(mode="json", exclude_none=True)

In [ ]:
#创建任务示例

import requests

def build_task_payload() -> dict:
    checklist = [
        ChecklistItem(text="read wiki", completed=True),
        ChecklistItem(text="write code"),
    ]

    task = TaskData(
        text="Update Habitica API Documentation - Tasks",
        type="todo",
        alias="hab-api-tasks",
        notes="Update the tasks api on GitHub",
        tags=["ed427623-9a69-4aac-9852-13deb9c190c3"],
        checklist=checklist,
        priority=2,
    )
    return task.to_dict()


def create_habitica_task(task_data: dict) -> dict:
    url = "https://habitica.com/api/v3/tasks/user"
    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
        "x-client": f"{HABITICA_USER_ID}-habitica-forge",
    }

    response = requests.post(url, headers=headers, json=task_data)
    response.raise_for_status()
    return response.json()


payload = build_task_payload()
result = create_habitica_task(payload)
print(result)

In [ ]:
# 删除任务示例
def delete_habitica_task(task_id: str) -> dict:
    url = f"https://habitica.com/api/v3/tasks/{task_id}"
    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
        "x-client": f"{HABITICA_USER_ID}-habitica-forge",
    }

    response = requests.delete(url, headers=headers)
    response.raise_for_status()
    return response.json()

# 示例调用
task_id_to_delete = "example-task-id"  # 替换为实际的任务 ID
delete_result = delete_habitica_task(task_id_to_delete)
print(delete_result)

# 删除任务的一个checklist项的示例
def delete_checklist_item(task_id: str, checklist_item_id: str) -> dict:
    url = f"https://habitica.com/api/v3/tasks/{task_id}/checklist/{checklist_item_id}"
    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
        "x-client": f"{HABITICA_USER_ID}-habitica-forge",
    }

    response = requests.delete(url, headers=headers)
    response.raise_for_status()
    return response.json()

# 删除任务的一个标签的示例
def delete_tag(task_id: str, tag_id: str) -> dict:
    url = f"https://habitica.com/api/v3/tasks/{task_id}/tags/{tag_id}"
    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
    }

In [3]:
#获取用户的所有任务数据示例

import requests

# 获取用户的所有任务数据
def get_habitica_tasks(task_type: Optional[str] = None) -> dict:
    url = "https://habitica.com/api/v3/tasks/user"
    params = {}
    if task_type not in [None, "habits", "dailys", "todos", "rewards", "completedTodos"]:
        raise ValueError("task_type的值必须是 None, 'habits', 'dailys', 'todos', 'rewards', 或 'completedTodos'")
    if task_type:
        params["type"] = task_type

    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
        "x-client": f"{HABITICA_USER_ID}-habitica-forge",
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

# 示例调用
tasks_metadata = get_habitica_tasks(task_type="todos")
tasks_metadata['data'] #会是一个列表，列表中的每个元素都是一个字典，包含任务数据

[{'_id': '6d8d96c5-58d0-41fa-9193-760f468123fe',
  'type': 'todo',
  'text': 'Update Habitica API Documentation - Tasks',
  'notes': 'Update the tasks api on GitHub',
  'alias': 'hab-api-tasks',
  'tags': ['ed427623-9a69-4aac-9852-13deb9c190c3',
   'a58c3d9d-c2a5-405b-a1eb-131dbc4c9375'],
  'value': 0,
  'priority': 2,
  'attribute': 'str',
  'challenge': {},
  'group': {'completedBy': {}, 'assignedUsers': []},
  'reminders': [],
  'byHabitica': False,
  'completed': False,
  'collapseChecklist': False,
  'checklist': [{'completed': True,
    'text': 'read wiki',
    'id': '7e159534-3faa-42de-881f-522d87c21dcb'},
   {'completed': False,
    'text': 'write code',
    'id': '0de7116d-cac2-40a2-8717-8f2e1bd808c1'}],
  'createdAt': '2026-03-11T10:23:50.841Z',
  'updatedAt': '2026-03-11T10:49:47.794Z',
  'userId': '480ac1e2-5e32-475d-bb29-71062e9b96b7',
  'id': '6d8d96c5-58d0-41fa-9193-760f468123fe'}]

In [ ]:
# 给任务添加标签示例
def add_tag_to_task(task_id: str, tag_id: str) -> dict:
    url = f"https://habitica.com/api/v3/tasks/{task_id}/tags/{tag_id}"
    headers = {
        "x-api-user": HABITICA_USER_ID,
        "x-api-key": HABITICA_API_TOKEN,
        "Content-Type": "application/json",
        "x-client": f"{HABITICA_USER_ID}-habitica-forge",
    }

    response = requests.post(url, headers=headers)
    response.raise_for_status()
    return response.json()

# 示例调用
task_id_to_update = "6d8d96c5-58d0-41fa-9193-760f468123fe"  # 替换为实际的任务 ID
tag_id_to_add = "a58c3d9d-c2a5-405b-a1eb-131dbc4c9375"  # 替换为实际的标签 ID
updated_task = add_tag_to_task(task_id_to_update, tag_id_to_add)
updated_task

{'success': True, 'data': {'challenge': {}, 'group': {'completedBy': {}, 'assignedUsers': []}, '_id': '6d8d96c5-58d0-41fa-9193-760f468123fe', 'type': 'todo', 'text': 'Update Habitica API Documentation - Tasks', 'notes': 'Update the tasks api on GitHub', 'alias': 'hab-api-tasks', 'tags': ['ed427623-9a69-4aac-9852-13deb9c190c3', 'a58c3d9d-c2a5-405b-a1eb-131dbc4c9375'], 'value': 0, 'priority': 2, 'attribute': 'str', 'reminders': [], 'byHabitica': False, 'completed': False, 'collapseChecklist': False, 'checklist': [{'completed': True, 'text': 'read wiki', 'id': '7e159534-3faa-42de-881f-522d87c21dcb'}, {'completed': False, 'text': 'write code', 'id': '0de7116d-cac2-40a2-8717-8f2e1bd808c1'}], 'createdAt': '2026-03-11T10:23:50.841Z', 'updatedAt': '2026-03-11T10:49:47.794Z', 'userId': '480ac1e2-5e32-475d-bb29-71062e9b96b7', 'id': '6d8d96c5-58d0-41fa-9193-760f468123fe'}, 'notifications': [{'type': 'ITEM_RECEIVED', 'data': {'icon': 'notif_armor_special_birthday2016', 'title': 'It’s Habitica’s Bi

In [ ]:
# 获取用户的所有标签示例
import requests
url = "https://habitica.com/api/v3/tags"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}

response = requests.get(url, headers=headers)
response.raise_for_status()
data = response.json()
data["data"]#返回的是一个列表，列表中的每个元素都是一个字典，包含标签id和name等数据

[{'id': 'ea5a6298-5ed3-4dcc-8ce7-978a8185a144', 'name': '优先级高'},
 {'id': 'd5ac8bfe-4155-47e4-b464-10b4a2424c37', 'name': '优先级中'},
 {'id': 'a58c3d9d-c2a5-405b-a1eb-131dbc4c9375', 'name': '优先级低'},
 {'id': 'f38df6f2-8865-483c-b776-c870058045ed', 'name': '长期任务'},
 {'id': '4dc7803b-3f4f-4171-9771-6efd652d5d00', 'name': '学习'}]

In [ ]:
# 创建新标签示例
import requests
url = "https://habitica.com/api/v3/tags"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}
data = {"name": "New Tag"}
response = requests.post(url, headers=headers, json=data)
data = response.json()
data["data"]#返回一个字典，包含新创建的标签的id和name等数据

In [ ]:
# 删除标签示例
import requests
tagId = "your_tag_id"
url = f"https://habitica.com/api/v3/tags/{tag_id}"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}
response = requests.delete(url, headers=headers)
data = response.json()

In [ ]:
# 更新标签示例
import requests
tagId = "your_tag_id"
url = f"https://habitica.com/api/v3/tags/{tagId}"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}
data = {"name": "Updated Tag Name"}
response = requests.put(url, headers=headers, json=data)
print(response.json())

In [ ]:
# 更新任务的某个checklist项的示例
task_id = "your task id"
item_id = "your item id"
url = f"https://habitica.com/api/v3/tasks/{task_id}/checklist/{item_id}"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}
# completed默认为False，即未完成
data = {"text": "new text","completed": False}
response = requests.put(url, headers=headers, json=data)
print(response.json())

In [4]:
# 更新任务示例
task_id = "6d8d96c5-58d0-41fa-9193-760f468123fe"
url = f"https://habitica.com/api/v3/tasks/{task_id}"
headers = {
    "x-api-user": HABITICA_USER_ID,
    "x-api-key": HABITICA_API_TOKEN,
    "Content-Type": "application/json",
    "x-client": f"{HABITICA_USER_ID}-habitica-forge",
}
# 比如我只需要更新text 和 notes
data = {"text": "new task text", "notes": "new task notes"}
response = requests.put(url, headers=headers, json=data)
print(response.json())

{'success': True, 'data': {'challenge': {}, 'group': {'completedBy': {}, 'assignedUsers': []}, '_id': '6d8d96c5-58d0-41fa-9193-760f468123fe', 'type': 'todo', 'text': 'new task text', 'notes': 'new task notes', 'alias': 'hab-api-tasks', 'tags': ['ed427623-9a69-4aac-9852-13deb9c190c3', 'a58c3d9d-c2a5-405b-a1eb-131dbc4c9375'], 'value': 0, 'priority': 2, 'attribute': 'str', 'reminders': [], 'byHabitica': False, 'completed': False, 'collapseChecklist': False, 'checklist': [{'completed': True, 'text': 'read wiki', 'id': '7e159534-3faa-42de-881f-522d87c21dcb'}, {'completed': False, 'text': 'write code', 'id': '0de7116d-cac2-40a2-8717-8f2e1bd808c1'}], 'createdAt': '2026-03-11T10:23:50.841Z', 'updatedAt': '2026-03-11T15:46:36.359Z', 'userId': '480ac1e2-5e32-475d-bb29-71062e9b96b7', 'id': '6d8d96c5-58d0-41fa-9193-760f468123fe'}, 'notifications': [{'type': 'ITEM_RECEIVED', 'data': {'icon': 'notif_armor_special_birthday2016', 'title': 'It’s Habitica’s Birthday!', 'text': 'Celebrate with us and enj